# Notebook 03: Add a New Problem Scaffold (DCC26)

Reference implementation of a minimal, reproducible benchmark problem scaffold.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

Use this as a pattern for structuring new benchmark problems with explicit contracts and validation checks.


## What makes a new problem benchmark-ready

Benchmark value comes from clarity and comparability, not only simulator sophistication.


In [ ]:
# Colab/local dependency bootstrap
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    !pip install engibench[beams2d] matplotlib gymnasium pybullet
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment). Set FORCE_INSTALL=True to install here.')


### Step 1 - Import scaffold dependencies

Keep imports minimal and interface-focused.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
from gymnasium import spaces

from engibench.constraint import bounded
from engibench.constraint import constraint
from engibench.core import ObjectiveDirection
from engibench.core import OptiStep
from engibench.core import Problem

import pybullet as p


### Step 2 - Implement PyBullet manipulator co-design problem contract

Ensure methods are deterministic and constraints/objectives are semantically explicit.


In [ ]:
class PlanarManipulatorCoDesignProblem(Problem[np.ndarray]):
    """Robotics co-design scaffold using a real PyBullet rollout loop."""

    version = 0
    objectives = (
        ("final_tracking_error_m", ObjectiveDirection.MINIMIZE),
        ("actuation_energy_j", ObjectiveDirection.MINIMIZE),
    )

    @dataclass
    class Conditions:
        target_x: Annotated[float, bounded(lower=0.20, upper=1.35)] = 0.85
        target_y: Annotated[float, bounded(lower=0.05, upper=1.20)] = 0.45
        payload_kg: Annotated[float, bounded(lower=0.0, upper=2.0)] = 0.8
        disturbance_scale: Annotated[float, bounded(lower=0.0, upper=0.30)] = 0.05

    @dataclass
    class Config(Conditions):
        sim_steps: Annotated[int, bounded(lower=60, upper=1200)] = 240
        dt: Annotated[float, bounded(lower=1e-4, upper=0.05)] = 1.0 / 120.0
        torque_limit: Annotated[float, bounded(lower=1.0, upper=50.0)] = 12.0
        max_iter: Annotated[int, bounded(lower=1, upper=300)] = 60

    dataset_id = "IDEALLab/planar_manipulator_codesign_v0"  # placeholder for future dataset integration
    container_id = None

    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(
            target_x=self.config.target_x,
            target_y=self.config.target_y,
            payload_kg=self.config.payload_kg,
            disturbance_scale=self.config.disturbance_scale,
        )

        # Design vector = [link1_m, link2_m, motor_strength, kp, kd, damping]
        self.design_space = spaces.Box(
            low=np.array([0.25, 0.20, 2.0, 5.0, 0.2, 0.0], dtype=np.float32),
            high=np.array([1.00, 0.95, 30.0, 120.0, 18.0, 1.5], dtype=np.float32),
            dtype=np.float32,
        )

        @constraint
        def reachable_workspace(design: np.ndarray, target_x: float, target_y: float, **_) -> None:
            l1, l2 = float(design[0]), float(design[1])
            r = float(np.sqrt(target_x**2 + target_y**2))
            assert l1 + l2 >= r + 0.03, f"target radius {r:.3f} exceeds reach {l1+l2:.3f}"

        @constraint
        def gain_consistency(design: np.ndarray, **_) -> None:
            kp, kd = float(design[3]), float(design[4])
            assert kd <= 2.2 * np.sqrt(max(kp, 1e-6)), f"kd={kd:.3f} too high for kp={kp:.3f}"

        self.design_constraints = [reachable_workspace, gain_consistency]

    def _build_robot(self, l1: float, l2: float, payload_kg: float, damping: float) -> tuple[int, int]:
        p.resetSimulation()
        p.setGravity(0, 0, -9.81)

        link_masses = [0.5 + 0.2 * payload_kg, 0.35 + 0.25 * payload_kg]
        link_collision = [-1, -1]
        link_visual = [
            p.createVisualShape(p.GEOM_CAPSULE, radius=0.025, length=l1, rgbaColor=[0.2, 0.5, 0.9, 1.0]),
            p.createVisualShape(p.GEOM_CAPSULE, radius=0.020, length=l2, rgbaColor=[0.9, 0.4, 0.2, 1.0]),
        ]
        qx = p.getQuaternionFromEuler([0.0, np.pi / 2.0, 0.0])

        robot = p.createMultiBody(
            baseMass=0.0,
            baseCollisionShapeIndex=-1,
            baseVisualShapeIndex=-1,
            basePosition=[0, 0, 0],
            linkMasses=link_masses,
            linkCollisionShapeIndices=link_collision,
            linkVisualShapeIndices=link_visual,
            linkPositions=[[0, 0, 0], [l1, 0, 0]],
            linkOrientations=[qx, qx],
            linkInertialFramePositions=[[l1 / 2.0, 0, 0], [l2 / 2.0, 0, 0]],
            linkInertialFrameOrientations=[[0, 0, 0, 1], [0, 0, 0, 1]],
            linkParentIndices=[0, 1],
            linkJointTypes=[p.JOINT_REVOLUTE, p.JOINT_REVOLUTE],
            linkJointAxis=[[0, 0, 1], [0, 0, 1]],
        )

        for j in [0, 1]:
            p.changeDynamics(robot, j, linearDamping=0.0, angularDamping=float(damping))

        return robot, 1

    def _inverse_kinematics_2link(self, x: float, y: float, l1: float, l2: float) -> tuple[float, float]:
        r2 = x * x + y * y
        c2 = (r2 - l1 * l1 - l2 * l2) / (2.0 * l1 * l2)
        c2 = float(np.clip(c2, -1.0, 1.0))
        s2 = float(np.sqrt(max(0.0, 1.0 - c2 * c2)))
        q2 = float(np.arctan2(s2, c2))
        q1 = float(np.arctan2(y, x) - np.arctan2(l2 * s2, l1 + l2 * c2))
        return q1, q2

    def _forward_kinematics_2link(self, q1: float, q2: float, l1: float, l2: float) -> tuple[float, float]:
        x = l1 * np.cos(q1) + l2 * np.cos(q1 + q2)
        y = l1 * np.sin(q1) + l2 * np.sin(q1 + q2)
        return float(x), float(y)

    def _rollout(self, design: np.ndarray, cfg: dict, return_trace: bool = False):
        l1, l2, motor_strength, kp, kd, damping = [float(v) for v in design]

        cid = p.connect(p.DIRECT)
        try:
            robot, _ = self._build_robot(l1, l2, cfg["payload_kg"], damping)
            q1_t, q2_t = self._inverse_kinematics_2link(cfg["target_x"], cfg["target_y"], l1, l2)

            err_trace = []
            tau_trace = []
            ee_trace = []
            energy = 0.0

            for _step in range(int(cfg["sim_steps"])):
                for j, q_t in enumerate([q1_t, q2_t]):
                    p.setJointMotorControl2(
                        bodyUniqueId=robot,
                        jointIndex=j,
                        controlMode=p.POSITION_CONTROL,
                        targetPosition=q_t,
                        positionGain=float(kp) / 120.0,
                        velocityGain=float(kd) / 50.0,
                        force=float(cfg["torque_limit"]) * float(motor_strength),
                    )

                if cfg["disturbance_scale"] > 0:
                    disturb = self.np_random.normal(0.0, cfg["disturbance_scale"], size=2)
                    p.applyExternalTorque(robot, 0, [0, 0, float(disturb[0])], p.LINK_FRAME)
                    p.applyExternalTorque(robot, 1, [0, 0, float(disturb[1])], p.LINK_FRAME)

                p.stepSimulation()

                js0 = p.getJointState(robot, 0)
                js1 = p.getJointState(robot, 1)
                q1, q2 = float(js0[0]), float(js1[0])
                dq1, dq2 = float(js0[1]), float(js1[1])
                tau1, tau2 = float(js0[3]), float(js1[3])

                ee_x, ee_y = self._forward_kinematics_2link(q1, q2, l1, l2)
                err = float(np.sqrt((ee_x - cfg["target_x"]) ** 2 + (ee_y - cfg["target_y"]) ** 2))

                err_trace.append(err)
                tau_trace.append((tau1, tau2))
                ee_trace.append((ee_x, ee_y))
                energy += (abs(tau1 * dq1) + abs(tau2 * dq2)) * float(cfg["dt"])

            final_error = float(err_trace[-1])
            obj = np.array([final_error, float(energy)], dtype=np.float32)

            if return_trace:
                trace = {
                    "ee_trace": np.array(ee_trace, dtype=np.float32),
                    "err_trace": np.array(err_trace, dtype=np.float32),
                    "tau_trace": np.array(tau_trace, dtype=np.float32),
                    "target": np.array([cfg["target_x"], cfg["target_y"]], dtype=np.float32),
                    "design": np.array(design, dtype=np.float32),
                    "objectives": obj,
                }
                return obj, trace

            return obj
        finally:
            p.disconnect(cid)

    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        cfg = {**self.config.__dict__, **(config or {})}
        x = np.clip(design.astype(np.float32), self.design_space.low, self.design_space.high)
        return self._rollout(x, cfg, return_trace=False)

    def optimize(self, starting_point: np.ndarray, config: dict | None = None):
        cfg = {**self.config.__dict__, **(config or {})}
        x = np.clip(starting_point.astype(np.float32), self.design_space.low, self.design_space.high)

        best = x.copy()
        best_obj = self.simulate(best, cfg)
        best_score = float(best_obj[0] + 0.02 * best_obj[1])

        history = [OptiStep(obj_values=best_obj, step=0)]
        step_scale = np.array([0.05, 0.05, 2.5, 8.0, 1.2, 0.08], dtype=np.float32)

        for step in range(1, int(cfg["max_iter"]) + 1):
            candidate = best + self.np_random.normal(0.0, 1.0, size=6).astype(np.float32) * step_scale
            candidate = np.clip(candidate, self.design_space.low, self.design_space.high)

            if self.check_constraints(candidate, cfg):
                history.append(OptiStep(obj_values=np.array([np.inf, np.inf], dtype=np.float32), step=step))
                continue

            obj = self.simulate(candidate, cfg)
            score = float(obj[0] + 0.02 * obj[1])
            if score < best_score:
                best, best_obj, best_score = candidate, obj, score

            history.append(OptiStep(obj_values=best_obj, step=step))

        return best, history

    def render(self, design: np.ndarray, *, open_window: bool = False):
        import matplotlib.pyplot as plt

        cfg = self.config.__dict__
        x = np.clip(design.astype(np.float32), self.design_space.low, self.design_space.high)
        obj, trace = self._rollout(x, cfg, return_trace=True)

        ee = trace["ee_trace"]
        err = trace["err_trace"]
        target = trace["target"]
        tau = trace["tau_trace"]

        fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))

        labels = ["link1", "link2", "motor", "kp", "kd", "damping"]
        axes[0].bar(labels, x, color=['#4c78a8', '#4c78a8', '#f58518', '#54a24b', '#e45756', '#72b7b2'])
        axes[0].set_title("Design variables")
        axes[0].tick_params(axis='x', rotation=35)

        axes[1].plot(ee[:, 0], ee[:, 1], lw=2, label="end-effector path")
        axes[1].scatter([target[0]], [target[1]], c='red', marker='x', s=70, label='target')
        r = x[0] + x[1]
        circle = plt.Circle((0, 0), r, color='gray', fill=False, linestyle='--', alpha=0.5)
        axes[1].add_patch(circle)
        axes[1].set_aspect('equal', 'box')
        axes[1].set_title("Task-space trajectory")
        axes[1].set_xlabel('x [m]')
        axes[1].set_ylabel('y [m]')
        axes[1].legend(fontsize=8)

        axes[2].plot(err, color='#e45756')
        axes[2].set_title("Tracking error over time")
        axes[2].set_xlabel("step")
        axes[2].set_ylabel("error [m]")
        axes[2].grid(alpha=0.3)

        axes[3].plot(np.abs(tau[:, 0]), label='|tau1|')
        axes[3].plot(np.abs(tau[:, 1]), label='|tau2|')
        axes[3].set_title("Actuation effort")
        axes[3].set_xlabel("step")
        axes[3].set_ylabel("torque [Nm]")
        axes[3].legend(fontsize=8)
        axes[3].grid(alpha=0.3)

        fig.suptitle(
            f"Objectives: final_error={obj[0]:.4f} m, energy={obj[1]:.3f} J",
            y=1.03,
        )
        fig.tight_layout()

        if open_window:
            plt.show()
        return fig, axes

    def random_design(self):
        d = self.np_random.uniform(self.design_space.low, self.design_space.high).astype(np.float32)
        return d, -1


### Step 3 - Smoke-test the scaffold

Validate behavior with simple checks before scaling to real domains.


Use the multi-panel render to read **where heat enters**, **how material is distributed**, and **where thermal bottlenecks remain**.


Use the final figure to interpret whether the design/controller combination reaches the target robustly with acceptable energy use.


In [ ]:
problem = PlanarManipulatorCoDesignProblem(
    seed=42,
    target_x=0.9,
    target_y=0.45,
    payload_kg=0.8,
    disturbance_scale=0.04,
    sim_steps=220,
    max_iter=40,
)
start, _ = problem.random_design()

cfg = {
    'target_x': 0.9,
    'target_y': 0.45,
    'payload_kg': 0.8,
    'disturbance_scale': 0.04,
    'sim_steps': 220,
    'dt': 1.0 / 120.0,
    'torque_limit': 12.0,
    'max_iter': 40,
}

print('design space:', problem.design_space)
print('objectives:', problem.objectives)
print('conditions:', problem.conditions)

viol = problem.check_constraints(start, config=cfg)
print('constraint violations:', len(viol))

obj0 = problem.simulate(start, config=cfg)
opt_design, history = problem.optimize(start, config=cfg)
objf = problem.simulate(opt_design, config=cfg)

print('initial objectives [tracking_error_m, energy_J]:', obj0.tolist())
print('final objectives   [tracking_error_m, energy_J]:', objf.tolist())
print('optimization steps:', len(history))
print('How to read plots: vars | task-space path | error timeline | torque timeline')

problem.render(opt_design)


## Mapping to real EngiBench contributions

Use this template to onboard new domains while preserving common evaluation semantics.


## Contribution checklist

Check for leakage risks, undocumented defaults, and missing reproducibility metadata before contribution.


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
